# DEM

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import geopandas as gpd
import numpy as np
import rasterio as rio
from pyproj import CRS
from shapely.geometry import box

In [ ]:
from etdataset.dem import (
    check_tiles,
    compute_egm96_height,
    copy_tiles,
    download_egm96_height,
    get_egm96_height,
)
from etdataset.interpolation import create_grid_array
from etdataset.utils import get_bbox_from_roi

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

## EGM96

In [ ]:
download_egm96_height()

In [ ]:
get_egm96_height(np.array([40.0, 42.0]), np.array([0.1, 0.0]))

In [ ]:
grid = create_grid_array(
    rio.coords.BoundingBox(left=-17.1, bottom=13.8, right=-15.9, top=15.0),
    crs=CRS(4326),
    resolution=0.1,
)

In [ ]:
grid

In [ ]:
egm96 = compute_egm96_height(grid)

In [ ]:
grid.plot()

In [ ]:
egm96.plot()

## Check DEM tiles

In [ ]:
output_dir = "out"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Create a shapely box from the bounding box
bbox = rio.coords.BoundingBox(-1.75, 42, 6.5, 45.5)
crs = CRS(4326)
geom = box(bbox.left, bbox.bottom, bbox.right, bbox.top)
gdf = gpd.GeoDataFrame({"geometry": [geom]}, crs=crs)
gdf.to_file(os.path.join(output_dir, "roi.shp"))

In [ ]:
roi_bbox, roi_crs = get_bbox_from_roi(os.path.join(output_dir, "roi.shp"))
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")
roi_bbox, roi_crs = get_bbox_from_roi(roi_path)

In [ ]:
if os.environ.get("MNT_PATH", None) is None:
    logging.error("Please set the variables MNT_PATH")

In [ ]:
mnt_path = os.path.join(os.environ["MNT_PATH"], "DEM_Copercinus_30m/")

In [ ]:
check_tiles(roi_bbox, roi_crs, base_dir=mnt_path)

In [ ]:
copy_tiles(roi_bbox, roi_crs, base_dir=mnt_path, output_dir=output_dir)